In [2]:
import torch
import torch.nn as nn

torch.manual_seed(0)

class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)

model = LinearModel()
print(model)

LinearModel(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)


In [3]:
for name, p in model.named_parameters():
    print(f"{name:16s} shape={tuple(p.shape)}  requires_grad={p.requires_grad}")

linear.weight    shape=(1, 1)  requires_grad=True
linear.bias      shape=(1,)  requires_grad=True


In [4]:
x = torch.linspace(-3, 3, 100).unsqueeze(1)   # (100, 1)
print("x     :", tuple(x.shape))
print("out   :", tuple(model(x).shape))

x     : (100, 1)
out   : (100, 1)


In [6]:
class A(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = torch.randn(1, requires_grad=True)
        self.b = torch.randn(1, requires_grad=True)
    def forward(self, x):
        return x * self.w + self.b

class B(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.randn(1))
        self.b = nn.Parameter(torch.randn(1))
    def forward(self, x):
        return x * self.w + self.b

In [7]:
a, b = A(), B()
print("A:", len(list(a.parameters()))) # expect : 0
print("B:", len(list(b.parameters()))) # expect : 1

A: 0
B: 2


In [8]:
loss_a = ((a(x) - (3 * x + 2)) ** 2).mean()
loss_a.backward()

print("a.w 는 텐서인가      :", type(a.w).__name__)
print("a.w.requires_grad   :", a.w.requires_grad)
print("a.w.grad            :", a.w.grad)          # ← 여기를 보세요
print("parameters() 개수    :", len(list(a.parameters())))

a.w 는 텐서인가      : Tensor
a.w.requires_grad   : True
a.w.grad            : tensor([-20.1419])
parameters() 개수    : 0


In [9]:
class C(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(1, 1), nn.Linear(1, 1)]        # 파이썬 list

class D(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(1, 1), nn.Linear(1, 1)])

print("C:", len(list(C().parameters())))
print("D:", len(list(D().parameters())))

C: 0
D: 4


In [ ]:
model.zero_grad()

with torch.no_grad():
          for p in model.parameters():
              p -= lr * p.grad

loss = ((pred - y) ** 2).mean()
loss.backward()


In [10]:
torch.manual_seed(0)
model = LinearModel()

x = torch.linspace(-3, 3, 100).unsqueeze(1)
y = 3 * x + 2
lr = 0.1

losses = []
for step in range(100):
    model.zero_grad()
    pred = pred = model(x)                       # (1)
    loss = loss = ((pred - y) ** 2).mean()                       # (2)
    loss.backward()
    with torch.no_grad():
        for p in model.parameters():
            p -= lr * p.grad                      # (3)
    losses.append(loss.item())

print("w =", model.linear.weight.item())
print("b =", model.linear.bias.item())
print("첫 손실 =", losses[0], " 마지막 손실 =", losses[-1])

w = 3.0
b = 1.999999761581421
첫 손실 = 29.82510757446289  마지막 손실 = 5.456968088664825e-14


# s4


In [11]:
sd = model.state_dict()
print(type(sd).__name__)
for k, v in sd.items():
    print(f"{k:20s} {tuple(v.shape)}  {v.flatten().tolist()}")

OrderedDict
linear.weight        (1, 1)  [3.0]
linear.bias          (1,)  [1.999999761581421]


In [12]:
class Nested(nn.Module):
    def __init__(self):
        super().__init__()
        self.block = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
    def forward(self, x):
        return self.block(x)

print(list(Nested().state_dict().keys()))

['block.0.weight', 'block.0.bias', 'block.2.weight', 'block.2.bias']


In [13]:
torch.save(model.state_dict(), "d1_model.pt")

model2 = LinearModel()
print("로드 전 최대 차이 :", (model2(x) - model(x)).abs().max().item())

ret = model2.load_state_dict(torch.load("d1_model.pt", weights_only=True))
print("반환값            :", ret)
print("반환값 타입        :", type(ret).__name__)
print("로드 후 최대 차이 :", (model2(x) - model(x)).abs().max().item())

로드 전 최대 차이 : 9.2404203414917
반환값            : <All keys matched successfully>
반환값 타입        : _IncompatibleKeys
로드 후 최대 차이 : 0.0


# s5


In [14]:
import gc, torch, torch.nn as nn

dev = "cuda"
x = torch.randn(65536, 1, device=dev)
y = 3 * x + 2
model = nn.Linear(1, 1).to(dev)
N = 300

def measure(fn):
    gc.collect(); torch.cuda.empty_cache()
    base = torch.cuda.memory_allocated()
    kept = fn()
    used = torch.cuda.memory_allocated() - base
    del kept
    gc.collect(); torch.cuda.empty_cache()
    return used / 1024**2

In [15]:
def case_a():                      # 텐서를 그대로 담는다
    out = []
    for _ in range(N):
        out.append(((model(x) - y) ** 2).mean())
    return out

def case_b():                      # .item() 으로 숫자만 꺼내 담는다
    out = []
    for _ in range(N):
        out.append(((model(x) - y) ** 2).mean().item())
    return out

def case_c():                      # no_grad 안에서 텐서를 그대로 담는다
    out = []
    with torch.no_grad():
        for _ in range(N):
            out.append(((model(x) - y) ** 2).mean())
    return out

for name, f in [("A  텐서 그대로", case_a),
                ("B  .item()", case_b),
                ("C  no_grad + 텐서", case_c)]:
    print(f"{name:22s} {measure(f):8.2f} MiB")

A  텐서 그대로                 83.27 MiB
B  .item()                 0.00 MiB
C  no_grad + 텐서            0.15 MiB


# c3

In [30]:
import torch, torch.nn as nn

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.zeros(1))
        self.b = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        return x * self.w + self.b

torch.manual_seed(0)
model = Model()
x = torch.linspace(-3, 3, 100).unsqueeze(1)
y = 3 * x + 2
lr = 0.1

for step in range(100):
    model.zero_grad()
    loss = ((model(x) - y) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        for p in model.parameters():
            p -= lr * p.grad

print("loss =", loss.item(), " w =", model.w.item(), " b =", model.b.item())

print(model.w)         # tensor([0.], requires_grad=True)   ← 있다
print(model.w.grad)    # tensor([-18.36])                   ← 기울기도 계산됐다
print(len(list(model.parameters())))   # 0                  ← 목록에만 없다


loss = 5.456968088664825e-14  w = 3.0  b = 1.999999761581421
Parameter containing:
tensor([3.], requires_grad=True)
tensor([1.5519e-07])
2


In [19]:
model2 = LinearModel()
model2.load_state_dict(torch.load("d1_model.pt", weights_only=True))

print(ret)
print(model2(x))

<All keys matched successfully>
tensor([[-7.0000],
        [-6.8182],
        [-6.6364],
        [-6.4545],
        [-6.2727],
        [-6.0909],
        [-5.9091],
        [-5.7273],
        [-5.5455],
        [-5.3636],
        [-5.1818],
        [-5.0000],
        [-4.8182],
        [-4.6364],
        [-4.4545],
        [-4.2727],
        [-4.0909],
        [-3.9091],
        [-3.7273],
        [-3.5455],
        [-3.3636],
        [-3.1818],
        [-3.0000],
        [-2.8182],
        [-2.6364],
        [-2.4545],
        [-2.2727],
        [-2.0909],
        [-1.9091],
        [-1.7273],
        [-1.5455],
        [-1.3636],
        [-1.1818],
        [-1.0000],
        [-0.8182],
        [-0.6364],
        [-0.4545],
        [-0.2727],
        [-0.0909],
        [ 0.0909],
        [ 0.2727],
        [ 0.4545],
        [ 0.6364],
        [ 0.8182],
        [ 1.0000],
        [ 1.1818],
        [ 1.3636],
        [ 1.5455],
        [ 1.7273],
        [ 1.9091],
        [ 2.0909],